In [1]:
import asyncio
import nest_asyncio
from tenacity import retry, wait_exponential, stop_after_attempt
import aiohttp
import pandas as pd

In [2]:
ObsData = pd.read_csv(r'../Transformed Data/RecordsClean.csv', sep=';', encoding="utf-8")
NativeData = pd.read_csv(r"../Transformed Data/NativeDataClean.csv", sep=";", encoding="utf-8")

In [3]:
ObsData

,Species,NAME_0,Realm,Cryptogenic,Dispersal,Eradicated,IntentionalRelease,Introduced,Established,ReportedFirstYear,Reference,ReferenceYear,AcceptedSpecies
0,Ostrinia nubilalis,Canada,Nearctic,NaN,NaN,0.0,NaN,NaN,1.0,NaN,CAB International (CABI) (2024). CABI Invasive...,2024,Ostrinia nubilalis
1,Zeiraphera diniana,Canada,Nearctic,NaN,NaN,0.0,NaN,NaN,1.0,NaN,CAB International (CABI) (2024). CABI Invasive...,2024,Zeiraphera griseana
2,Plutella xylostella,Canada,Nearctic,NaN,NaN,0.0,NaN,NaN,1.0,NaN,CAB International (CABI) (2024). CABI Invasive...,2024,Plutella xylostella
3,Helicoverpa armigera,Brazil,Neotropical,NaN,NaN,0.0,NaN,NaN,1.0,NaN,CAB International (CABI) (2024). CABI Invasive...,2024,Helicoverpa armigera
4,Spodoptera exempta,Seychelles,Oceanina,NaN,NaN,0.0,NaN,NaN,1.0,NaN,CAB International (CABI) (2024). CABI Invasive...,2024,Spodoptera exempta
...,...,...,...,...,...,...,...,...,...,...,...,...,...
17440,Luthrodes pandava,Japan,Sino-Japanese,0.0,0.0,0.0,0.0,1.0,1.0,NaN,"Wu, L., et al. (2010). Elucidating genetic sig...",2010,Luthrodes pandava
17441,Luthrodes pandava,Madagascar,Madagascan,0.0,0.0,0.0,0.0,1.0,1.0,NaN,"Wu, L., et al. (2010). Elucidating genetic sig...",2010,Luthrodes pandava
17442,Luthrodes pandava,South Korea,Palearctic,0.0,0.0,0.0,0.0,1.0,1.0,NaN,"Wu, L., et al. (2010). Elucidating genetic sig...",2010,Luthrodes pandava
17443,Parapoynx diminutalis,United States,Nearctic,0.0,NaN,NaN,0.0,1.0,1.0,1976.0,"Buckingham, G., Bennett, C. (1996). Laboratory...",1996,Parapoynx diminutalis


# 1. Code

## 1.1. GBIF Family Extraction

In [4]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def GBIF_Family(session, species):
    url = f"https://api.gbif.org/v1/species/match?name={species}"
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get('family'):
                return data.get('family')
            else:
                return None
    
    except Exception as e:
        print(f"Request failed for species '{species}': {e}")
        return None

async def GBIF_Family_Sessions(species_list):
    async with aiohttp.ClientSession() as session:
        tasks = [GBIF_Family(session, species) for species in species_list]
        return await asyncio.gather(*tasks)

def GBIF_Family_Extract(species_list):
    return asyncio.get_event_loop().run_until_complete(GBIF_Family_Sessions(species_list))

## 1.2. Lepidoptera Crosscheck

In [5]:
nest_asyncio.apply()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5))
async def GBIF_Lepidoptera(session, family):
    
    url = f"https://api.gbif.org/v1/species/match?name={family}"
    try:
        async with session.get(url) as response:
            response.raise_for_status()
            data = await response.json()
            
            if data.get('order') == 'Lepidoptera':
                return 1
            elif data.get('order') is None:
                return None
            else:
                return 0
            
    except Exception as e:
        print(f"Error fetching data for {family}: {e}")
        return False

async def GBIF_Lepidoptera_Sessions(family_list):
    async with aiohttp.ClientSession() as session:
        tasks = [GBIF_Lepidoptera(session, family) for family in family_list]
        return await asyncio.gather(*tasks)

def GBIF_Lepidoptera_Extract(family_list):
    return asyncio.get_event_loop().run_until_complete(GBIF_Lepidoptera_Sessions(family_list))

# 2. Extraction Taxonomy

In [6]:
TaxonomyData = ObsData[['Species', 'AcceptedSpecies']].copy().drop_duplicates().reset_index(drop=True)

In [7]:
TaxonomyData = pd.concat([TaxonomyData, 
                          pd.DataFrame({
                              'Species': list(set(TaxonomyData['AcceptedSpecies']) - set(TaxonomyData['Species'])),
                              'AcceptedSpecies': list(set(TaxonomyData['AcceptedSpecies']) - set(TaxonomyData['Species']))})], 
                         ignore_index=True).drop_duplicates().reset_index(drop=True)


In [8]:
TaxonomyData['Genus'] = TaxonomyData['AcceptedSpecies'].str.split(' ').str[0]
TaxonomyData['Family'] = GBIF_Family_Extract(TaxonomyData['AcceptedSpecies'])
TaxonomyData['Lepidoptera'] = GBIF_Lepidoptera_Extract(TaxonomyData['Family'])

## 2.2. Manual Updates

In [9]:
TaxonomyData[(TaxonomyData['Lepidoptera']==0)|(TaxonomyData['Lepidoptera'].isnull())]

,Species,AcceptedSpecies,Genus,Family,Lepidoptera
142,Antheraea pernyi,Antheraea pernyi,Antheraea,Saturniidae,NaN
306,Saturnia japonica,Saturnia japonica,Saturnia,Saturniidae,NaN
525,Hyalophora euryalus,Hyalophora euryalus,Hyalophora,Saturniidae,NaN
684,Samia cynthia,Samia cynthia,Samia,Saturniidae,NaN
810,Homoeographa lamceolella,Homoeographa lanceolella,Homoeographa,None,NaN
847,Actias selene,Actias selene,Actias,Saturniidae,NaN
853,Antheraea yamamai,Antheraea yamamai,Antheraea,Saturniidae,NaN
857,Antheraea paphia,Antheraea paphia,Antheraea,Saturniidae,NaN
858,Antheraea polyphemus,Antheraea polyphemus,Antheraea,Saturniidae,NaN
880,Calephelis virginiensis,Calephelis virginiensis,Calephelis,None,NaN


In [10]:
TaxonomyData.loc[TaxonomyData['Genus'] == 'Homoeographa', 'Family'] = 'Pyralidae'
TaxonomyData.loc[TaxonomyData['Genus'] == 'Calephelis', 'Family'] = 'Riodinidae'

As Saturniidae, Pyralidae and Riodinidae are all Lepidopterans and the remaining have been identified as such the cross-check column can be dropped as it is a constant for all.

In [11]:
TaxonomyData.drop(columns=['Lepidoptera'], inplace=True)

In [12]:
TaxonomyData

,Species,AcceptedSpecies,Genus,Family
0,Ostrinia nubilalis,Ostrinia nubilalis,Ostrinia,Crambidae
1,Zeiraphera diniana,Zeiraphera griseana,Zeiraphera,Tortricidae
2,Plutella xylostella,Plutella xylostella,Plutella,Plutellidae
3,Helicoverpa armigera,Helicoverpa armigera,Helicoverpa,Noctuidae
4,Spodoptera exempta,Spodoptera exempta,Spodoptera,Noctuidae
...,...,...,...,...
1465,Clethrogyna turbata,Clethrogyna turbata,Clethrogyna,Erebidae
1466,Elkalyce argiades,Elkalyce argiades,Elkalyce,Lycaenidae
1467,Zale minerea,Zale minerea,Zale,Erebidae
1468,Notioplusia illustrata,Notioplusia illustrata,Notioplusia,Noctuidae


In [13]:
TaxonomyData.to_csv(r'../Transformed Data/TaxonomyClean.csv', index=False, sep=';', encoding='utf-8')

# 3. Extraction References

In [14]:
References = pd.concat([NativeData[["Reference", "ReferenceYear"]].copy(), ObsData[["Reference", "ReferenceYear"]].copy()])

In [15]:
References["Reference Year"] = References["ReferenceYear"].astype(int)
References.drop(columns="ReferenceYear", inplace=True)
References.drop_duplicates(inplace=True)

In [16]:
References.reset_index(drop=True, inplace=True)
References.to_csv(r"../Transformed Data/ReferencesClean.csv", sep=";", encoding="utf-8", index=False)